# 03 — OCR extraction (L0 baseline)

**L0 = pretrained EasyOCR + per-field allow-lists + empty-cell skip.** No vocab
snapping (L1) and no db matching (L2) — those are later notebooks.

This notebook is a thin driver. All the logic lives in **`src/ocr.py`** so the
Streamlit app can import the exact same code path. Here we just:

1. load `field_map.json` (field order + types + comb cells) and the nb-02
   `preprocess_report.json` (checkbox reads),
2. run `ocr.extract_form` over all 100 forms in `data/scans/crops/`,
3. write `data/generated/predictions_l0.csv` aligned to `labels.csv` columns,
4. score rough per-field accuracy vs `labels.csv` so we have a baseline.

> Crops are already cleaned/trimmed grayscale PNGs (nb 02). We only *read* them.
> First run downloads the ~64 MB EasyOCR `english_g2` model. CPU is fine.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# repo root = parent of notebooks/ ; make src/ importable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import ocr  # src/ocr.py

CROPS_ROOT  = ROOT / "data" / "scans" / "crops"
FIELD_MAP   = ROOT / "data" / "template" / "field_map.json"
REPORT      = ROOT / "data" / "scans" / "preprocess_report.json"
LABELS      = ROOT / "data" / "scans" / "labels.csv"
OUT_CSV     = ROOT / "data" / "generated" / "predictions_l0.csv"

print("repo root :", ROOT)
print("crops     :", CROPS_ROOT, "->", len(list(CROPS_ROOT.glob("P*"))), "form folders")

repo root : c:\Projects\form-ocr_v2
crops     : c:\Projects\form-ocr_v2\data\scans\crops -> 100 form folders


In [2]:
# Load the field map (canonical field order) and the nb-02 checkbox reads.
field_map = ocr.load_field_map(str(FIELD_MAP))
report    = ocr.load_checkbox_report(str(REPORT))

FIELD_ORDER = list(field_map["fields"].keys())
print(len(FIELD_ORDER), "fields:", FIELD_ORDER)
print("checkbox report covers", len(report), "forms")

# Build the EasyOCR reader ONCE (heavy). No CUDA on the target machine -> CPU.
GPU = False                      # set True only if a CUDA GPU is available
reader = ocr.get_reader(gpu=GPU) # downloads english_g2 on first call
print("EasyOCR reader ready (gpu=%s)" % GPU)

20 fields: ['last_name', 'first_name', 'address', 'email', 'department', 'date_of_birth', 'age', 'gender', 'ssn', 'insurance_number', 'phone_number', 'blood_type', 'date_of_visit', 'doctor_name', 'chief_complaint', 'medical_history', 'allergies', 'current_medications', 'emergency_contact_name', 'emergency_contact_phone']
checkbox report covers 100 forms


Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR reader ready (gpu=False)


In [3]:
%%time
# Run L0 over all 100 forms. ~minutes on CPU (per-cell digit OCR is the bulk).
rows = ocr.predict_all(str(CROPS_ROOT), field_map, report, gpu=GPU, progress=True)
print("extracted", len(rows), "forms")

c:\Projects\form-ocr_v2\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [  1/100] P0001
  [  2/100] P0002
  [  3/100] P0003
  [  4/100] P0004
  [  5/100] P0005
  [  6/100] P0006
  [  7/100] P0007
  [  8/100] P0008
  [  9/100] P0009
  [ 10/100] P0010
  [ 11/100] P0011
  [ 12/100] P0012
  [ 13/100] P0013
  [ 14/100] P0014
  [ 15/100] P0015
  [ 16/100] P0016
  [ 17/100] P0017
  [ 18/100] P0018
  [ 19/100] P0019
  [ 20/100] P0020
  [ 21/100] P0021
  [ 22/100] P0022
  [ 23/100] P0023
  [ 24/100] P0024
  [ 25/100] P0025
  [ 26/100] P0026
  [ 27/100] P0027
  [ 28/100] P0028
  [ 29/100] P0029
  [ 30/100] P0030
  [ 31/100] P0031
  [ 32/100] P0032
  [ 33/100] P0033
  [ 34/100] P0034
  [ 35/100] P0035
  [ 36/100] P0036
  [ 37/100] P0037
  [ 38/100] P0038
  [ 39/100] P0039
  [ 40/100] P0040
  [ 41/100] P0041
  [ 42/100] P0042
  [ 43/100] P0043
  [ 44/100] P0044
  [ 45/100] P0045
  [ 46/100] P0046
  [ 47/100] P0047
  [ 48/100] P0048
  [ 49/100] P0049
  [ 50/100] P0050
  [ 51/100] P0051
  [ 52/100] P0052
  [ 53/100] P0053
  [ 54/100] P0054
  [ 55/100] P0055
  [ 56/100

In [4]:
# Assemble the predictions table, columns aligned to labels.csv (minus `image`).
labels = pd.read_csv(LABELS, dtype=str).fillna("")
label_cols = [c for c in labels.columns if c != "image"]   # patient_id + 20 fields

pred = pd.DataFrame(rows)
# guarantee every expected column exists and is ordered like labels.csv
for c in label_cols:
    if c not in pred.columns:
        pred[c] = ""
pred = pred[label_cols].fillna("")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pred.to_csv(OUT_CSV, index=False)
print("wrote", OUT_CSV)
pred.head()

wrote c:\Projects\form-ocr_v2\data\generated\predictions_l0.csv


,patient_id,last_name,first_name,address,email,department,date_of_birth,age,gender,ssn,...,phone_number,blood_type,date_of_visit,doctor_name,chief_complaint,medical_history,allergies,current_medications,emergency_contact_name,emergency_contact_phone
0,P0001,Brewer,O,45G2 Elm BLvc Fuburv PA 23392,moabrukc425gmlcom,Rheumatlogy,/2/22,0,Male,--25,...,-2-,B+,//2,Ds . Nccarthy,Fregueut urnaton,No,Nonc bnown,NJone,Altssen' Portcr,4--25
1,P0002,Hopbivs,Wojciech,"3462 Oak Dr,Franklin O4 1RGts",in2f2i24l.bfa.e2811n,Psychiaty,//,,Other,--,...,-2-,AB-,//92,Dr Antunzs,Difficulty Swallowng,Nlonq,SaMe Mile,Noue,Qecepalf Ma ye,5-9-9
2,P0003,TCaey,Jclew-,5383 Qinc Ln FranLlin NY 694t0,ieleuatfacy46pmton.me,Ophthal mology,9//5,,Other,2--,...,18-9-2,AB+,/5/2,Seruais,Weight loss,Nony,Po lev,Qamipril 5 ma,Pomoin Clecc,52-43-9
3,P0004,Guener,AlarTha,1184 Chureh Ct Sprinqtictel FU 33451,,IJa seular Surgery,//,,Male,21-1-8,...,-65-,A-,8//22,A Gaua n,Coxhson,Nov,Nne Enonn,Akuc,Annc Rtit,-23-35
4,P0005,Tuzfme k,Ann,4360 Elm uy Fairvie wJ Oh 4063t,ann.tuzimekialouoaom,Gaskacatero Ipgq,3//,5,Male,-2-2,...,3--,A+,//,Dc . oa Gnha,Dizz ihess,"Diabets, Heart Drskase Asthma",ftoruastakn,None,Kayla IGm,--9


## L0 accuracy baseline

Exact-match per field (case-insensitive, whitespace-trimmed) against
`labels.csv`. This is a deliberately strict L0 yardstick — a single wrong
digit fails the whole field. L1 (vocab snap) and L2 (db match) are expected to
lift the fuzzy text + identity fields well above these numbers.


In [5]:
def norm(s):
    return " ".join(str(s).strip().split()).lower()

# align on patient_id
gt = labels.set_index("patient_id")
pr = pred.set_index("patient_id")
common = [p for p in gt.index if p in pr.index]
fields = [c for c in label_cols if c != "patient_id"]

per_field = {}
for f in fields:
    correct = sum(norm(gt.loc[p, f]) == norm(pr.loc[p, f]) for p in common)
    per_field[f] = correct / len(common)

acc = (pd.Series(per_field).sort_values(ascending=False)
       .rename("accuracy").to_frame())
acc["accuracy"] = (acc["accuracy"] * 100).round(1)
print(f"L0 exact-match accuracy over {len(common)} forms (%):\n")
print(acc.to_string())
print(f"\nMACRO mean over fields : {acc['accuracy'].mean():.1f}%")

# overall cell-level accuracy
total = len(common) * len(fields)
hit = sum(norm(gt.loc[p, f]) == norm(pr.loc[p, f]) for p in common for f in fields)
print(f"MICRO cell accuracy    : {100*hit/total:.1f}%  ({hit}/{total} cells)")

L0 exact-match accuracy over 100 forms (%):

                         accuracy
blood_type                   99.0
gender                       99.0
first_name                   33.0
last_name                    24.0
age                          12.0
chief_complaint               7.0
allergies                     6.0
department                    6.0
medical_history               2.0
emergency_contact_name        2.0
doctor_name                   1.0
address                       0.0
email                         0.0
insurance_number              0.0
ssn                           0.0
date_of_birth                 0.0
phone_number                  0.0
date_of_visit                 0.0
current_medications           0.0
emergency_contact_phone       0.0

MACRO mean over fields : 14.6%
MICRO cell accuracy    : 14.6%  (291/2000 cells)


In [6]:
# Quick error peek: a few mismatches per field to eyeball L0 failure modes.
import itertools
for f in fields:
    misses = [(p, gt.loc[p, f], pr.loc[p, f]) for p in common
              if norm(gt.loc[p, f]) != norm(pr.loc[p, f])]
    if misses:
        print(f"\n[{f}] {len(misses)} misses (gt -> pred):")
        for p, g, pd_ in itertools.islice(misses, 3):
            print(f"   {p}: {g!r} -> {pd_!r}")


[last_name] 76 misses (gt -> pred):
   P0002: 'Hopkins' -> 'Hopbivs'
   P0003: 'Tracy' -> 'TCaey'
   P0004: 'Guerrero' -> 'Guener'

[first_name] 67 misses (gt -> pred):
   P0001: 'Moa' -> 'O'
   P0003: 'Jelena' -> 'Jclew-'
   P0004: 'Martha' -> 'AlarTha'

[address] 100 misses (gt -> pred):
   P0001: '4562 Elm Blvd, Auburn, PA 23396' -> '45G2 Elm BLvc Fuburv PA 23392'
   P0002: '3762 Oak Dr, Franklin, OH 18675' -> '3462 Oak Dr,Franklin O4 1RGts'
   P0003: '5583 Pine Ln, Franklin, NY 69470' -> '5383 Qinc Ln FranLlin NY 694t0'

[email] 100 misses (gt -> pred):
   P0001: 'moa.brewer12@gmail.com' -> 'moabrukc425gmlcom'
   P0002: 'wojciech.hopkins28@outlook.com' -> 'in2f2i24l.bfa.e2811n'
   P0003: 'jelena.tracy1@proton.me' -> 'ieleuatfacy46pmton.me'

[department] 94 misses (gt -> pred):
   P0001: 'Rheumatology' -> 'Rheumatlogy'
   P0002: 'Psychiatry' -> 'Psychiaty'
   P0003: 'Ophthalmology' -> 'Ophthal mology'

[date_of_birth] 100 misses (gt -> pred):
   P0001: '01/02/2022' -> '/2/22'
   P0